### Let's fix some of the error handling for getting the variants

This is the python code we want to replicate
```
    def get_variants(self):
        '''Read variants from input VCF file, yield one at a time'''
        for input_row in self.reader: 
            self.total_variants_in_input += 1
            if is_valid_input_row(input_row):
                this_ref = input_row["ref"]
                # allow possibly multiple alts in the same variant, split them into separate alleles
                alts = input_row["alt"].split(",")
                for this_alt in alts:
                    this_var_type = get_var_type(this_ref, this_alt)
                    if is_acceptable_variant(dict(input_row), self.varclass, this_var_type, this_ref, this_alt, self.max_indel_size):
                        output_row = copy(input_row)
                        output_row["alt"] = this_alt
                        output_row["pos"] = int(input_row["pos"])
                        output_row["vartype"] = this_var_type
                        self.num_variants_analysed += 1
                        yield output_row
            else:
                logging.warning(f"Skipping invalid input row: {dict(input_row)}")

    def log_totals(self):
        self.num_variants_skipped = self.total_variants_in_input - self.num_variants_analysed
        logging.info(f"Total variants in input: {self.total_variants_in_input}")
        logging.info(f"Num variants kept for analysis: {self.num_variants_analysed}")
        logging.info(f"Num variants skipped: {self.num_variants_skipped}")
```
```
# Check if an input variant is acceptable for this analysis and valid
# The purpose of this method is to reject and thus skup any input variants
# that we cannot intepret within the current analysis
def is_acceptable_variant(row, varclass, vartype, ref, alt, max_indel_size):
    if not is_only_DNA_bases(ref) or not is_only_DNA_bases(alt):
        # ref and alt must only consist of DNA bases
        logging.info(f"Skipping variant with non-DNA bases: {row}")
        return False
    if not is_desired_type(varclass, vartype):
        # this particular variant must be of a type that is compatible
        # with the varclass specified on the command line, (SNV, INDEL(
        logging.info(f"Skipping variant of unwanted type: {vartype} {row}")
        return False
    if not is_within_max_size(varclass, max_indel_size, ref, alt):
        # The variant must not be longer than the maximum size, if specified
        logging.info(f"Skipping variant that is too large {row}")
        return False
    if varclass == "INDEL" and not is_valid_indel(ref, alt):
        # Check INDELs for appropriate formatting
        logging.info(f"Skipping invalid INDEL: {row}")
        return False
    return True
    
    
VALID_DNA_BASES = set("ATGC")
    
def is_only_DNA_bases(sequence):
    return set(sequence.upper()).issubset(VALID_DNA_BASES) 


# Check that an input variant is within some size bound. This is normally
# only relevant for INDELs where the max_indel_size can be set on
# the command line. However, for completeness we also check that SNVs
# are indeed the expected size of 1 DNA base.
def is_within_max_size(varclass, max_indel_size, ref, alt):
    if varclass == "SNV":
        return len(ref) == 1 and len(alt) == 1
    elif varclass == "INDEL" and max_indel_size is not None:
        this_indel_size = abs(len(ref) - len(alt))
        return this_indel_size <= max_indel_size
    return True

# Check that the INDEL is specified in a way that we can interpet:
# there must be at least 1 context base, the shortest of ref and alt
# must be a prefix of the other. They must not have the same length.
def is_valid_indel(ref, alt):
    ref_len = len(ref)
    alt_len = len(alt)
    if ref_len < alt_len:
        shortest = ref
        longest = alt
    elif alt_len < ref_len:
        shortest = alt
        longest = ref
    else:
        # ref and alt are the same length, not currently supported 
        return False
    return ref_len >= 1 and alt_len >= 1 and longest.startswith(shortest)

REQUIRED_INPUT_VARIANT_FIELDS = set(["chrom", "pos", "ref", "alt"])

def is_valid_input_row(row):
    return set(row.keys()).issuperset(REQUIRED_INPUT_VARIANT_FIELDS)

def vcf_reader(file):
    for line in file:
        if line.startswith('#'):
            continue
        fields = line.strip().split()
        # Technically VCF requires the first 8 fields to be defined, but we want to be as liberal
        # as possible in accepting inputs.
        if len(fields) >= 5:
            chrom, pos, _id, ref, alt = fields[:5]
            yield {"chrom": chrom, "pos": pos, "ref": ref, "alt": alt}
        else:
            logging.warning(f"Skipping input row: {line}")
```

This is our current rust code

- We shouldn't use bcf reader as we already know which of the fields we need, instead of loading everything into c structures
    - htslib bcf is also designed for bcf instead of vcf, so there will be runtime penalty for parsing into vcf
- We could use [noodles](https://github.com/zaeleus/noodles) which is a rust bioinformatics I/O crate
    - However we want to minimize the use of external dependencies

In [ ]:
use std::fs::File;
use std::io::{BufRead, BufReader};
use std::collections::VecDeque;

// Change to bcf::Reader? so can handle gzip files?
fn vcf_reader(file_path: &str, varclass: &str) -> Result<VecDeque<Variant>, Box<dyn Error>> {
    let file = File::open(file_path)?;
    let reader = BufReader::new(file);

	let mut variants = VecDeque::new();

    for line_result in reader.lines() {
        let line = line_result?;
        if line.starts_with("#") {
            continue;
            }
        
        let fields: Vec<&str> = line.split_whitespace().collect();
        
        if fields.len() >= 5 {
            let chrom = fields[0].to_string();

            let pos = match fields[1].parse::<u64>() {
                Ok(p) => p,
                Err(_) => {
                    eprintln!("Warning: invalid POS, skipping row: {}", line);
                    continue;
                }
            };

            let refr = fields[3].to_string();

            for alt in fields[4].split(',') {
                let vartype = get_var_type(&refr, alt);

                match varclass.to_ascii_uppercase().as_str() {
                    "SNV" => {
                        if matches!(vartype, VarType::Snv) {
                            variants.push_back(Variant {
                                chrom: chrom.clone(),
                                pos,
                                refr: refr.clone(),
                                alt: alt.to_string(),
                                vartype,
                                features: LocusFeatures::Snv(LocusFeaturesSNV::default()),
                            });
                        }
                    }
                    "INDEL" => {
                        if matches!(vartype, VarType::Ins | VarType::Del) {
                            variants.push_back(Variant {
                                chrom: chrom.clone(),
                                pos,
                                refr: refr.clone(),
                                alt: alt.to_string(),
                                vartype,
                                features: LocusFeatures::Indel(LocusFeaturesINDEL::default()),
                            });
                        }
                    }
                    _ => continue,
                }
            }
        } else {
            eprintln!("Warning: Skipping input row: {}", line);
        }
    }
			
	Ok(variants)
}

#[derive(Debug, Clone, Copy)]
enum VarType {
    Snv,
    Del,
    Ins,
    Unknown,
}

impl VarType {
    fn as_str(&self) -> &'static str {
        match self {
            VarType::Snv => "SNV",
            VarType::Del => "DEL",
            VarType::Ins => "INS",
            VarType::Unknown => "UNKNOWN",            
        }
    }
}

fn get_var_type(refr: &str, alt: & str) -> VarType {
    if refr.len() == 1 && alt.len() == 1 {
        VarType::Snv
    } else if refr.len() > alt.len() {
        VarType::Del
    } else if refr.len() < alt.len() {
        VarType::Ins
    } else {
        eprintln!(
            "Warning: Cannot determine the type of variant with ref: {} and alt: {}",
            refr, alt
        );
        VarType::Unknown
    }
}

We can check for acceptable variants and deal with delins here before the processing the read features for INDELs (see [here](https://github.com/bjpop/varlap/blob/f4a48c801187d0427caba1010dbfc623eb0d49b4/varlap/varlap.py#L623))

Delins/indels use the left-most base not changed in the variant as a reference allele. For example:
```
#CHROM  POS  ID  REF  ALT
chr1    10   .   TAC  TGTT
```
This represents a AC deletion, followed by a GTT insertion. Therefore, we can make our assumptions:

- For delins, ref and alt are both > 1 (We cannot say for sure it is a delin; we must first check whether the reference allele is the same.)
- For snvs, ref and alt are both 1
- For deletions, ref is > 1, alt is 1
- For insertions, ref is 1, alt is > 1

**Note: We are assuming that the sites are biallelic instead of multiallelic**

For example, if the VCF is not normalized using something like `bcftools norm --multiallelics -both --fasta-ref ${REFERENCE_FASTA}`, we can have representations such as 
```
#CHROM POS ID REF ALT QUAL FILTER INFO
20 3 . C G . PASS DP=100
20 2 . TC T . PASS DP=100
20 2 . TC TCA . PASS DP=100
```
`TC -> TCA` is a insertion of A to the reference allele TC. Our algorithm would then filter this out.

Another example below showing before and after of decomposition and left-alignment of a multiallelic VCF:
- https://re-docs.genomicsengland.co.uk/aggv3_site_qc_decomposition/#__tabbed_1_1

Other references:
- https://www.biostars.org/p/9571257/#9571460
- https://samtools.github.io/bcftools/bcftools.html#norm
- https://re-docs.genomicsengland.co.uk/variant_normalisation/

varlap currently has a check for valid indels (below), which assumes that the above `TC -> TCA` conversion may be represented in the VCF/CSV/TSV. If so, we should move the DELIN check to where it is specified by Bernie. However, since we are processing multiallelic sites (since we are iterating over an `alt` vec that has been split by `,`, how do we deal with SNVs in multiallelic sites if there is the ref and alt are > 1 due to many dels/ins?)

```
# Check that the INDEL is specified in a way that we can interpet:
# there must be at least 1 context base, the shortest of ref and alt
# must be a prefix of the other. They must not have the same length.
def is_valid_indel(ref, alt):
    ref_len = len(ref)
    alt_len = len(alt)
    if ref_len < alt_len:
        shortest = ref
        longest = alt
    elif alt_len < ref_len:
        shortest = alt
        longest = ref
    else:
        # ref and alt are the same length, not currently supported 
        return False
    return ref_len >= 1 and alt_len >= 1 and longest.startswith(shortest)
```

#### Updating how we determine the variant type (biallelic assumption)

In [ ]:
fn get_var_type(refr: &str, alt: & str) -> VarType {
    if refr.len() == 1 && alt.len() == 1 {
        VarType::Snv
    } else if refr.len() > 1 && alt.len() > 1 {
        eprintln!(
            "Warning: Cannot determine the type of variant with ref: {} and alt: {}",
            refr, alt
        );
        VarType::Unknown
    } else if refr.len() > alt.len() {
        VarType::Del
    } else if refr.len() < alt.len() {
        VarType::Ins
    } else {
        eprintln!(
            "Warning: Cannot determine the type of variant with ref: {} and alt: {}",
            refr, alt
        );
        VarType::Unknown
    }
}

#### Validating input row for VCF
We can have a small check for validating whether the VCF parsed is valid
- Since the header line starts with `#` instead of the optional fields `##` we can split it into fields and check whether they match the format `["#CHROM", "POS", "ID", "REF", "ALT"]`

In [ ]:
fn is_valid_vcf_header_line(line: &str) -> bool {
    let expected = ["#CHROM", "POS", "ID", "REF", "ALT"];
    line.split_whitespace().take(5).eq(expected.into_iter())
}

fn vcf_reader(file_path: &str, varclass: &str) -> Result<VecDeque<Variant>, Box<dyn Error>> {
    let file = File::open(file_path)?;
    let reader = BufReader::new(file);

	let mut variants = VecDeque::new();

    for line_result in reader.lines() {
        let line = line_result?;
        
        if line.starts_with("##") {
            continue;
        }

        if line.starts_with("#") && !is_valid_vcf_header_line(&line) {
            return Err("Invalid VCF header".into());
        }

        let fields: Vec<&str> = line.split_whitespace().collect();

Instead of using std::error:Error, we can use [anyhow](https://docs.rs/anyhow/latest/anyhow/)

However, because we want the program to run on HPCs, we want there to be relevant Exit Codes; `anyhow` will default all exit codes to be `1`. Therefore, we could also use `thiserror` in conjunction with `anyhow` to also provide custom exit codes.

Python varlap uses:
```
EXIT_FILE_IO_ERROR = 1 
EXIT_COMMAND_LINE_ERROR = 2 
EXIT_BAD_FILE_FORMAT = 3
```
`clap` CLI already handles exit code 2; 



In [ ]:
// NOTE: This assumes that the variants from only one chromosome
fn get_chrom_info(variants: &VecDeque<Variant>) -> Option<ChromInfo> {
    let first = variants.front()?;
    let last = variants.back()?;

    Some(ChromInfo {
        min_pos: first.pos,
        max_pos: last.pos,
    })
}

For this code snippet, we should change `Option` to just a return `ChromInfo`. As our variant parser will only create a new chromosome struct if we have a valid variant, a chromsome should always have at least 1 variant which will be able to call `.front()` and `.back()`. Therefore, if this is not the case, it means that there is a bug in our code. Therefore, we can use anyhow `.expect()`, which will cause a panic, stopping execution.

In [ ]:
fn get_chrom_info(variants: &VecDeque<Variant>) -> ChromInfo {
    let first = variants
        .front()
        .expect("Internal error: chromosome contains no variants");

    let last = variants
        .back()
        .expect("Internal error: chromosome contains no variants");

    ChromInfo {
        min_pos: first.pos,
        max_pos: last.pos,
    }
}

https://www.reddit.com/r/rust/comments/wtu5te/how_should_i_propagate_my_errors_to_include_a/

https://burntsushi.net/rust-error-handling/#advice-for-library-writers

https://www.lpalmieri.com/posts/error-handling-rust/#anyhow-or-thiserror

https://dev.to/codeprototype/taking-the-unhappy-path-with-result-option-unwrap-and-operator-in-rust-482b

Typically, the recommended approach for dealing with errors is to use anyhow for applications and thiserror for libraries; anyhow makes it so that you don't have to specify the error in the Result (e.g. from Result<T, std::io::Error> to Result<T>, which is shorthand for Result<T, anyhow::Error>). however, it also means that it erases an specific error type that may have been returned. I.e. all errors just turn into anyhow::Error type. ThisError simplifies the process of creating an enum with all your different error types you want to document.

I decided on a combo of Anyhow and ThisError. As we are creating a CLI application and not a library, a user does not have detailed error types which they can handle themselves. However, as per Bernies notes on building software for HPCs, we want custom exit status values as different programs are typically included in workflows, and so custom exit status values can quickly identify for users what type of error happened and how they can deal with it. 

Wanted custom error type for I/O errors, which we set as Status Error 2.

Each submodule a main function that calls other unit functions; these unit functions can be for a singular process such as procesing/parsing inputs for example. If there is a type of I/O error, we set the error to a custom error type of enum (AppError) using thiserror. This unit function will return our custom error type (AppError). Then in the main function, we can use the `?` operator when calling the function as it returns a result. We then use anyhow in this main function, which will convert the AppError into an anyhow error (as our main function will also return an anyhow error for general errors). In our main module, which calls each of the other submodules, we can then catch any errors that have propagated/bubbled up. We can then use the `downcast_ref` method on the error type, which converts the generic `Any` trait that is implemented by `anyhow` error handling back into the specifc error type (`AppError`) that we defined using `thiserror` (if it was originally an `AppError`).

Essentially we are propagating/bubbling up errors from `Unit::AppError (thiserror)` -> `SubmoduleMain::Anyhow (anyhow)` -> `Main` catch error and downcast if available.

We can then match this error code to an error submodule that we have defined with custom error codes and correspondingly return an associated status exit code and custom error message